<a href="https://colab.research.google.com/github/JosephBigDataAnalytics/JKaremera-Programming-BigDataAnalytics/blob/main/Task_15_model_1_complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Model 1- predicting policy category for complaint

Task: Fine-tune based on the data 'credit_card_aa.csv'. The data is also available at https://huggingface.co/datasets/priyaannamani/credit_card_qa Create two fine-tuned models. The first one should take complaint as the input and predict policy_category. The second should take complaint as the input and predict resolution. Take a sample of 60 records to fine-tune and evaluate the fine-tuned models on all the records in the dataset. You can use either or both distilbert_base_uncased and distilgpt2 models. Performance comparative analysis of how well fine-tuning works. Write a short narrative at the end to discuss your insights from fine-tuning. Generate python code explaining each step to execute the task

step 1- import libraires

In [ ]:
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import warnings
warnings.filterwarnings("ignore")

!pip install evaluate
import evaluate
from sklearn.metrics import classification_report, confusion_matrix

print("✅ Libraries imported successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
✅ Libraries imported successfully.


# STEP 2: Load Dataset Directly from Hugging Face Hub

In [ ]:
print("\n📦 Loading dataset from Hugging Face Hub...")

raw_dataset = load_dataset("priyaannamani/credit_card_qa", split="train")

# Convert to Pandas for easier inspection
df_full = raw_dataset.to_pandas()

print(f"✅ Dataset loaded. Total records: {len(df_full)}")
print(f"Columns: {df_full.columns.tolist()}")
print(df_full.head(3))


📦 Loading dataset from Hugging Face Hub...


README.md:   0%|          | 0.00/435 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/22.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/80 [00:00<?, ? examples/s]

✅ Dataset loaded. Total records: 80
Columns: ['complaint', 'relevant_policy', 'policy_category', 'resolution', 'validity']
                                           complaint  \
0  Dana Wu, using card ending 0044, purchased $12...   
1  Dana Wu is asking why she didn't get 4% cashba...   
2  Dana Wu purchased $50 worth of electronics alo...   

                                     relevant_policy policy_category  \
0  You will earn 4% Cash Back on the first $25,00...   Cashback - 4%   
1  You will earn 4% Cash Back on the first $25,00...   Cashback - 4%   
2  Some merchants may sell these products/service...   Cashback - 4%   

                                          resolution  \
0  Apply missing 4% cashback for the eligible gro...   
1  Explain that wholesale clubs are often not cla...   
2  Explain that only the grocery portion of the p...   

                                            validity  
0  Valid: Purchase was at an eligible merchant an...  
1  Invalid: Merchant not cla

# STEP 3: Inspect and Prepare Target Labels

In [ ]:
# Identify the correct column names (complaint text + category)
# Based on the dataset: 'complaint' → input, 'policy_category' → label

TEXT_COL = "complaint"          # Input feature
LABEL_COL = "policy_category"  # Target label

print(f"\n🏷️  Unique policy categories ({df_full[LABEL_COL].nunique()}):")
print(df_full[LABEL_COL].value_counts())

# Encode string labels to integers
label_list = sorted(df_full[LABEL_COL].unique().tolist())
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

df_full["label"] = df_full[LABEL_COL].map(label2id)
print(f"\n✅ Label encoding complete. Mapping: {label2id}")


🏷️  Unique policy categories (21):
policy_category
Cashback - 4%                                20
Purchase Security                            20
Cashback - Exclusions                         7
Contact Information - Insurance               5
Cashback - 2%                                 3
Contact Information - Protection Services     3
Contact Information - Banking                 2
Cashback - 2% Gas                             2
Contact Information - Emergency Services      2
Contact Information                           2
Cashback - Redemption                         2
Cashback - Eligible Categories                2
Contact Information - Concierge               2
Cashback - Foreign Transactions               1
Cashback - 1% Groceries                       1
Cashback - Calculation                        1
Cashback - Expiration                         1
Miscellaneous - Program Termination           1
Cashback - Posting Timeline                   1
Miscellaneous - Program Changes     

# STEP 4: Sample 60 Records for Fine-Tuning
# (Stratified sample to balance classes where possible)

In [ ]:
# Stratified sampling for balanced class representation
df_sample = (
    df_full
    .groupby(LABEL_COL, group_keys=False)
    .apply(lambda x: x.sample(
        min(len(x), max(1, int(60 * len(x) / len(df_full)))),
        random_state=42
    ))
)

# Ensure exactly 60 records
if len(df_sample) < 60:
    remaining = df_full.drop(df_sample.index).sample(60 - len(df_sample), random_state=42)
    df_sample = pd.concat([df_sample, remaining])
elif len(df_sample) > 60:
    df_sample = df_sample.sample(60, random_state=42)

df_sample = df_sample[[TEXT_COL, "label"]].reset_index(drop=True)
print(f"✅ Training sample size: {len(df_sample)}")
print(f"Class distribution in sample:\n{df_sample['label'].value_counts()}")

✅ Training sample size: 60
Class distribution in sample:
label
4     16
20    15
7      6
16     4
2      2
17     2
3      1
0      1
1      1
9      1
5      1
6      1
8      1
12     1
11     1
10     1
13     1
15     1
14     1
18     1
19     1
Name: count, dtype: int64


# STEP 5: Split Sample into Train / Validation (80/20)

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_val = train_test_split(
    df_sample, test_size=0.2, random_state=42,
)

print(f"\n✅ Train size: {len(df_train)} | Validation size: {len(df_val)}")

# Convert to Hugging Face Dataset objects
hf_train = Dataset.from_pandas(df_train.reset_index(drop=True))
hf_val   = Dataset.from_pandas(df_val.reset_index(drop=True))



✅ Train size: 48 | Validation size: 12


# STEP 6: Load Tokenizer and Tokenize the Data

In [ ]:
MODEL_NAME = "distilbert-base-uncased"

print(f"\n🔤 Loading tokenizer: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    """Tokenize complaint text with truncation and padding."""
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=256,   # Complaints can be long; 256 tokens balances speed & context
        padding=False,    # Dynamic padding handled by DataCollator
    )

# Apply tokenization
hf_train = hf_train.map(tokenize_function, batched=True)
hf_val   = hf_val.map(tokenize_function, batched=True)

# Set format for PyTorch
hf_train = hf_train.remove_columns([TEXT_COL])
hf_val   = hf_val.remove_columns([TEXT_COL])

print("✅ Tokenization complete.")
print(f"Train features: {hf_train.column_names}")



🔤 Loading tokenizer: distilbert-base-uncased...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

✅ Tokenization complete.
Train features: ['label', 'input_ids', 'token_type_ids', 'attention_mask']


 STEP 7: Load Pre-trained DistilBERT Classification Model

In [ ]:
num_labels = len(label_list)

print(f"\n🤖 Loading {MODEL_NAME} with {num_labels} output labels...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)
print("✅ Model loaded with classification head attached.")



🤖 Loading distilbert-base-uncased with 21 output labels...


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded with classification head attached.


# STEP 8: Define Metrics for Evaluation

In [ ]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    """Compute accuracy during training evaluation."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)


# STEP 9: Configure Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./credit_card_finetuned",   # Directory to save checkpoints
    num_train_epochs=5,                      # More epochs compensate for small sample
    per_device_train_batch_size=8,           # Batch size for training
    per_device_eval_batch_size=8,            # Batch size for evaluation
    learning_rate=2e-5,                      # Standard fine-tuning LR for BERT variants
    weight_decay=0.01,                       # Regularization to prevent overfitting
    eval_strategy="epoch",                   # Evaluate after each epoch
    save_strategy="epoch",                   # Save checkpoint each epoch
    load_best_model_at_end=True,             # Auto-load best checkpoint at end
    metric_for_best_model="accuracy",        # Use accuracy to pick best model
    logging_dir="./logs",                    # TensorBoard logs
    logging_steps=10,
    report_to="none",                        # Disable W&B / external logging
    seed=42,
)

print("\n✅ Training arguments configured.")


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



✅ Training arguments configured.


# STEP 10: Initialize Trainer and Fine-Tune

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\n🏋️  Starting fine-tuning on 60-record sample...")
trainer.train()
print("✅ Fine-tuning complete!")


🏋️  Starting fine-tuning on 60-record sample...


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,2.822860,0.500000
2,2.952947,2.638976,0.666667
3,2.952947,2.496850,0.666667
4,2.696692,2.407241,0.666667
5,2.505733,2.376333,0.666667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


✅ Fine-tuning complete!


  # STEP 11: Evaluate on ALL Records in the Full Dataset


In [ ]:
print(f"\n📊 Evaluating fine-tuned model on ALL {len(df_full)} records...")

# Prepare the full dataset for inference
df_eval = df_full[[TEXT_COL, "label"]].copy().reset_index(drop=True)
hf_eval_full = Dataset.from_pandas(df_eval)

# Tokenize full evaluation set
hf_eval_full = hf_eval_full.map(tokenize_function, batched=True)
hf_eval_full = hf_eval_full.remove_columns([TEXT_COL])

# Run predictions
predictions_output = trainer.predict(hf_eval_full)
predicted_labels   = np.argmax(predictions_output.predictions, axis=-1)
true_labels        = predictions_output.label_ids



📊 Evaluating fine-tuned model on ALL 80 records...


Map:   0%|          | 0/80 [00:00<?, ? examples/s]

# STEP 12: Print Full Evaluation Report

In [ ]:
overall_accuracy = np.mean(predicted_labels == true_labels)
print(f"\n🎯 Overall Accuracy on Full Dataset: {overall_accuracy:.4f} ({overall_accuracy*100:.2f}%)")

# Detailed per-class report
target_names = [id2label[i] for i in range(num_labels)]
print("\n📋 Classification Report (Full Dataset):")
print(classification_report(
    true_labels,
    predicted_labels,
    target_names=target_names,
    zero_division=0
))

# Confusion Matrix
cm = confusion_matrix(true_labels, predicted_labels)
print("\n🔢 Confusion Matrix:")
cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)
print(cm_df)



🎯 Overall Accuracy on Full Dataset: 0.5000 (50.00%)

📋 Classification Report (Full Dataset):
                                           precision    recall  f1-score   support

                                 Cashback       0.00      0.00      0.00         1
                  Cashback - 1% Groceries       0.00      0.00      0.00         1
                            Cashback - 2%       0.00      0.00      0.00         3
                        Cashback - 2% Gas       0.00      0.00      0.00         2
                            Cashback - 4%       0.33      1.00      0.50        20
                   Cashback - Calculation       0.00      0.00      0.00         1
           Cashback - Eligible Categories       0.00      0.00      0.00         2
                    Cashback - Exclusions       0.00      0.00      0.00         7
                    Cashback - Expiration       0.00      0.00      0.00         1
          Cashback - Foreign Transactions       0.00      0.00      0.00   

# STEP 13: Save the Fine-Tuned Model Locally

In [ ]:
model.save_pretrained("./credit_card_finetuned_final")
tokenizer.save_pretrained("./credit_card_finetuned_final")
print("\n💾 Model saved to ./credit_card_finetuned_final")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


💾 Model saved to ./credit_card_finetuned_final


In [ ]:
print("📦 Creating zip archive of the fine-tuned model...")
!zip -r credit_card_finetuned_model.zip credit_card_finetuned_final/

print("✅ Zip archive created. Verifying...")
!ls -lh


📦 Creating zip archive of the fine-tuned model...
  adding: credit_card_finetuned_final/ (stored 0%)
  adding: credit_card_finetuned_final/tokenizer_config.json (deflated 42%)
  adding: credit_card_finetuned_final/tokenizer.json (deflated 71%)
  adding: credit_card_finetuned_final/model.safetensors (deflated 8%)
  adding: credit_card_finetuned_final/config.json (deflated 67%)
✅ Zip archive created. Verifying...
total 236M
drwxr-xr-x 7 root root 4.0K Mar 11 12:16 credit_card_finetuned
drwxr-xr-x 2 root root 4.0K Mar 11 12:18 credit_card_finetuned_final
-rw-r--r-- 1 root root 236M Mar 11 12:21 credit_card_finetuned_model.zip
drwxr-xr-x 1 root root 4.0K Jan 16 14:24 sample_data


# STEP 14: Inference on New Complaints (Example Usage)

In [ ]:
from transformers import pipeline

print("\n🔍 Testing inference on sample complaints...")
classifier = pipeline(
    "text-classification",
    model="./credit_card_finetuned_final",
    tokenizer="./credit_card_finetuned_final",
    device=-1   # Use CPU; change to 0 for GPU
)

sample_complaints = [
    "I was charged an unexpected fee on my credit card without any prior notice.",
    "My credit card was used fraudulently and the bank refused to reverse the charges.",
    "I cannot access my account online and customer service has not resolved the issue.",
]

for complaint in sample_complaints:
    result = classifier(complaint, truncation=True, max_length=256)
    print(f"  Complaint: {complaint[:60]}...")
    print(f"  → Predicted Category: {result[0]['label']} (score: {result[0]['score']:.3f})\n")


🔍 Testing inference on sample complaints...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

  Complaint: I was charged an unexpected fee on my credit card without an...
  → Predicted Category: Cashback - 4% (score: 0.071)

  Complaint: My credit card was used fraudulently and the bank refused to...
  → Predicted Category: Cashback - 4% (score: 0.075)

  Complaint: I cannot access my account online and customer service has n...
  → Predicted Category: Cashback - 4% (score: 0.072)

